In [0]:
# Databricks notebook source

# --------------------------------------------
# Project : NYC Taxi Lakehouse
# Layer   : Bronze
# Purpose : Load raw Yellow Taxi data into Delta Lake
# Source  : NYC TLC Yellow Taxi Trip Records
# --------------------------------------------

In [0]:
from pyspark.sql import functions as F

In [0]:
# Source file
RAW_FILE = "/Volumes/taxi/default/raw_files/yellow_tripdata_2026-01.parquet"

# Target table
BRONZE_TABLE = "taxi.bronze.yellow_taxi"

In [0]:
bronze_df = spark.read.parquet(RAW_FILE)

In [0]:
display(bronze_df)

In [0]:
bronze_df.printSchema()

In [0]:
print(f"Total Rows : {bronze_df.count()}")
print(f"Total Columns : {len(bronze_df.columns)}")

In [0]:
display(
    bronze_df.select(
        F.min("tpep_pickup_datetime").alias("Minimum Pickup Time"),
        F.max("tpep_pickup_datetime").alias("Maximum Pickup Time")
    )
)

In [0]:
display(
    bronze_df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in bronze_df.columns
    ])
)

In [0]:
(
    bronze_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(BRONZE_TABLE)
)

In [0]:
display(
    spark.table(BRONZE_TABLE)
)

In [0]:
print(f"Bronze Table Rows : {spark.table(BRONZE_TABLE).count():,}")

In [0]:
%sql
DESCRIBE DETAIL taxi.bronze.yellow_taxi;